# 04 · Results and the choice of the final model

1. Scores of every run: mean, SD and minimum over the 5 outer test folds, on **all test cells** (measured + filled; the main result, as V3) and on **measured cells only**.
2. The final model by V3's rule, decided before the results:
   * eligible: mean Clay and Sand recall ≥ 0.50, and no fold below 0.25;
   * best = highest mean Macro-F1;
   * among eligible models within one standard error of the best: lowest fold SD of Macro-F1, then lowest Brier score.
3. Scores by distance to the nearest training borehole, block-size sensitivity and purge.

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
OUT = H.out_dir(S, "04_results")
main_block = H.block_sizes(S)[0]
folds_all = H.fold_scores(preds)
summ = H.summary(folds_all)
summ_m = H.summary(H.fold_scores(preds[preds.measured == 1]))
folds_all.to_csv(OUT / "metrics_by_outer_fold.csv", index=False)
summ.to_csv(OUT / "metrics_fold_mean_sd.csv", index=False)
summ_m.to_csv(OUT / "metrics_fold_mean_sd_measured_only.csv", index=False)
show = ["model", "macro_f1_mean", "macro_f1_sd", "balanced_accuracy_mean", "recall_clay_mean", "recall_sand_mean",
        "roc_auc_mean", "brier_mean"]
for name, s in (("all test cells", summ), ("measured cells only", summ_m)):
    print(f"Main block size ({main_block} m), {name}:")
    display(s[(s.block_m == main_block) & (s.purge_m == 0)][show].sort_values("macro_f1_mean", ascending=False).round(3))

## 1. The final model (V3's rule)

In [ ]:
sel = H.select_model(summ, main_block)
display(sel["table"][["model", "eligible", "macro_f1_mean", "macro_f1_sd", "recall_clay_min", "recall_sand_min",
                      "brier_mean"]].round(3).sort_values("macro_f1_mean", ascending=False))
print("Highest mean Macro-F1:", sel["highest_mean"], f"({sel['best_macro_f1_mean']:.3f}) | one SE = {sel['one_se']:.3f}")
print("Within one SE:", sel["within_1se"])
print("FINAL MODEL:", sel["selected"])
print("(same rule on measured cells only:", H.select_model(summ_m, main_block)["selected"], ")")
_ = (OUT / "selected_model.json").write_text(json.dumps({k: v for k, v in sel.items() if k != "table"}, indent=1, default=str))

## 2. By distance to the nearest training borehole

In [ ]:
dist = H.distance_table(preds[(preds.block_m == main_block) & (preds.purge_m == 0)])
dist.to_csv(OUT / "metrics_by_distance_to_training.csv", index=False)
display(dist.pivot_table(index="distance_bin", columns="model", values="macro_f1", observed=True).round(3))
display(dist.groupby("distance_bin", observed=True).n.max().rename("test cells per model"))

## 3. Block-size sensitivity and purge

In [ ]:
rob = summ[summ.purge_m == 0].pivot_table(index="model", columns="block_m", values="macro_f1_mean")
rob.to_csv(OUT / "macro_f1_by_block_size.csv"); display(rob.round(3))
pur = summ[summ.block_m == main_block].pivot_table(index="model", columns="purge_m", values="macro_f1_mean")
pur.to_csv(OUT / "macro_f1_by_purge_distance.csv"); display(pur.round(3))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 05.")